# BRFSS Pipeline BI Analysis

This notebook provides Business Intelligence analysis for the BRFSS Big Data Analytics pipeline results.

## Sections:
1. Load and explore processed data
2. Analyze pipeline metrics
3. Create visualizations
4. Model performance insights
5. Interactive dashboards

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Load config
config_path = Path("pipeline_config.json")
if config_path.exists():
    with open(config_path, 'r') as f:
        config = json.load(f)
else:
    config = {
        "selected_csv": "data/processed/selected_columns.csv",
        "metrics_path": "outputs/metrics/metrics.json",
        "spark_metrics_path": "outputs/metrics/spark_metrics.json",
        "plot_dir": "outputs/metrics/plots"
    }

print("Configuration loaded:")
for key, value in config.items():
    print(f"{key}: {value}")

In [ ]:
# Load processed data
data_path = Path(config["selected_csv"])
if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"✅ Loaded processed data: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\n📊 Data Preview:")
    display(df.head())
    
    print("\n📈 Basic Statistics:")
    display(df.describe())
    
    print("\n🔍 Data Types:")
    display(df.dtypes)
else:
    print("❌ Processed data not found. Please run the pipeline first.")
    df = None

## 2. Pipeline Metrics Analysis

In [ ]:
# Load pipeline metrics
metrics_path = Path(config["metrics_path"])
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print("📊 Pipeline Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
else:
    print("❌ Pipeline metrics not found")
    metrics = {}

# Load Spark metrics
spark_metrics_path = Path(config["spark_metrics_path"])
if spark_metrics_path.exists():
    with open(spark_metrics_path, 'r') as f:
        spark_metrics = json.load(f)
    print("\n🔥 Spark Metrics:")
    for key, value in spark_metrics.items():
        print(f"  {key}: {value}")
else:
    print("❌ Spark metrics not found")
    spark_metrics = {}

## 3. Data Visualizations

In [ ]:
if df is not None:
    try:
        # Distribution plots
        numeric_cols = df.select_dtypes(include=[np.number]).columns[:4]  # First 4 numeric columns
        
        if len(numeric_cols) > 0:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            fig.suptitle('Data Distributions', fontsize=16)
            
            for i, col in enumerate(numeric_cols):
                ax = axes[i//2, i%2]
                sns.histplot(df[col], ax=ax, kde=True)
                ax.set_title(f'Distribution of {col}')
            
            plt.tight_layout()
            plt.show()
        else:
            print("No numeric columns found for distribution plots")
            
        # Correlation heatmap
        if len(numeric_cols) > 1:
            plt.figure(figsize=(10, 8))
            corr = df[numeric_cols].corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
            plt.title('Correlation Heatmap')
            plt.show()
            
    except Exception as e:
        print(f"❌ Visualization error: {e}")
        print("💡 Try using Plotly for interactive plots or check matplotlib installation")
else:
    print("No data available for visualizations")

## 4. Model Performance Insights

In [ ]:
# Analyze model performance
if metrics:
    print("🎯 Model Performance Analysis:")
    
    # Classification metrics
    if 'accuracy' in metrics:
        acc = metrics['accuracy']
        print(f"Accuracy: {acc:.3f} ({'Good' if acc > 0.8 else 'Needs improvement'})")
    
    if 'precision' in metrics:
        prec = metrics['precision']
        print(f"Precision: {prec:.3f}")
        
    if 'recall' in metrics:
        rec = metrics['recall']
        print(f"Recall: {rec:.3f}")
        
    if 'f1_score' in metrics:
        f1 = metrics['f1_score']
        print(f"F1 Score: {f1:.3f}")
        
    # Overall assessment
    if 'accuracy' in metrics and metrics['accuracy'] > 0.75:
        print("✅ Model performance is satisfactory")
    else:
        print("⚠️ Model may need improvement")
        
else:
    print("No metrics available for analysis")

# Check existing plots
plot_dir = Path(config["plot_dir"])
if plot_dir.exists():
    plots = list(plot_dir.glob("*.png"))
    print(f"\n📈 Existing pipeline plots ({len(plots)} files):")
    for plot in plots:
        print(f"  - {plot.name}")
else:
    print("No pipeline plots found")

## 5. Summary and Recommendations

This notebook provides a comprehensive BI analysis of the BRFSS pipeline results.

### Key Findings:
- Data processing status
- Model performance metrics
- Visualization of key insights

### Recommendations:
- Run the full pipeline to generate processed data and metrics
- Use this notebook for regular BI reporting
- Consider adding more advanced visualizations (interactive plots, dashboards)
- Monitor model performance over time